[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/autoencoder_vae/autoencoder_vae.ipynb)

In [ ]:
import os
import sys
import subprocess

# Default Colab GPU preference: A100 (fallback: L4)
if "google.colab" in sys.modules and not os.path.exists("models.py"):
    if not os.path.exists("vision_transformers"):
        subprocess.run(
            ["git", "clone", "https://github.com/a-ramakrishnan/vision_transformers.git"],
            check=True,
        )
    os.chdir("vision_transformers")
    if not os.path.exists("autoencoder_vae/models.py"):
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "checkout", "feat/modular-structure"], check=True)
    os.chdir("autoencoder_vae")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [ ]:
import torch
from dataset import get_mnist_dataloaders
from models import Autoencoder, VAE
from train import train_autoencoder, train_vae, save_training_results
from visualize import plot_reconstructions, plot_vae_samples

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f"Using device: {device} ({gpu_name})")

BATCH_SIZE = 128
train_loader, test_loader = get_mnist_dataloaders(batch_size=BATCH_SIZE)

In [ ]:
# =====================================================================
# PART 1: Train & Visualize Vanilla Autoencoder
# =====================================================================
AE_EPOCHS = 10
AE_LR = 1e-3
AE_LATENT_DIM = 2

ae_model = Autoencoder(latent_dim=AE_LATENT_DIM).to(device)
ae_model, ae_history = train_autoencoder(
    ae_model,
    train_loader,
    epochs=AE_EPOCHS,
    lr=AE_LR,
    device=device,
    checkpoint_path="autoencoder_mnist.pth",
)

plot_reconstructions(
    ae_model,
    test_loader,
    device=device,
    is_vae=False,
    title="Vanilla Autoencoder: Original (Top) vs Reconstructed (Bottom)",
    save_path="ae_reconstructions.png",
)

In [ ]:
# =====================================================================
# PART 2: Train & Visualize Variational Autoencoder (VAE)
# =====================================================================
VAE_EPOCHS = 15
VAE_LR = 1e-3
VAE_LATENT_DIM = 20

vae_model = VAE(input_dim=784, hidden_dim=400, latent_dim=VAE_LATENT_DIM).to(device)
vae_model, vae_history = train_vae(
    vae_model,
    train_loader,
    epochs=VAE_EPOCHS,
    lr=VAE_LR,
    device=device,
    checkpoint_path="vae_mnist.pth",
)

save_training_results(ae_history, vae_history, save_path="training_results.json")

plot_reconstructions(
    vae_model,
    test_loader,
    device=device,
    is_vae=True,
    title="VAE: Original (Top) vs Reconstructed (Bottom)",
    save_path="vae_reconstructions.png",
)
plot_vae_samples(
    vae_model,
    latent_dim=VAE_LATENT_DIM,
    num_samples=16,
    device=device,
    save_path="vae_samples.png",
)